In [2]:
from dotenv import load_dotenv 
import os 

load_dotenv()

True

In [3]:
import os 

DB_URI = os.getenv("DB_URI")

In [4]:
#@markdown Check Table

import psycopg2

DB_URI = DB_URI

conn = psycopg2.connect(DB_URI)
cur = conn.cursor()

cur.execute("""
    SELECT table_name 
    FROM information_schema.tables
    WHERE table_schema = 'public';
""")

tables = cur.fetchall()

print("Tables:")
for table in tables:
    print(table[0])

cur.close()
conn.close()

Tables:
checkpoint_migrations
checkpoints
checkpoint_blobs
checkpoint_writes


In [21]:
#@markdown Delete Tables
import psycopg

DB_URI = DB_URI

with psycopg.connect(DB_URI, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            DO $$ DECLARE
                r RECORD;
            BEGIN
                -- Drop all tables
                FOR r IN (SELECT tablename FROM pg_tables WHERE schemaname = 'public') LOOP
                    EXECUTE 'DROP TABLE IF EXISTS ' || quote_ident(r.tablename) || ' CASCADE';
                END LOOP;
            END $$;
        """)
        print("All tables deleted successfully.")

All tables deleted successfully.


In [1]:
import psycopg

user_text = "playing"

In [4]:
from sqlalchemy import create_engine, inspect, text

DB_URI = DB_URI

engine = create_engine(DB_URI)
inspector = inspect(engine)

tables = inspector.get_table_names()

for table in tables:
    print(f"\n===== TABLE: {table} =====")
    
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT * FROM {table} LIMIT 5"))
        rows = result.fetchall()
        
        if rows:
            for row in rows:
                print(row)
        else:
            print("No data.")

In [8]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage, HumanMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    
    agent2_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_2"
    ]
    

    agent3_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_3_as_imposter"
    ]
    
    agent4_words = [
        msg.content for msg in request.messages
        if isinstance(msg, HumanMessage) and getattr(msg, "name", None) == "agent_4"
    ]
    
    #print("Agent2:", agent2_words)

    #print(request.messages)

   # print("Latest", words)

    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are a player in an Imposter social deduction game.
                    Players' words so far: {words}
                    Your secret keyword is {content}: 

                    Rules:
                    - Answer in ONE word only.
                    - Directly describe what it is or how it is used.
                    - Do NOT repeat previous words from other players.
                    - Be subtle. Do not make it too obvious.
                    """

    elif round_c in ["round_2", "round_3"]:
        return f"""
                Players' words so far: {words}
                Your keyword: {content}
                Think carefully:
                - Then give ONE new word.
                - Do NOT repeat any previous word.
                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_2's responses:
            {agent2_words}

            Player_3's responses:
            {agent3_words}

            Player_4's responses:
            {agent4_words}

            Based on these responses:
            - Guess who is the imposter (Player_2 or Player_3 or Player_4).
            - Return only the player name.
            """
    else:
        pass 
    return round_c
DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_1",
        checkpointer=checkpointer,
    )
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Ronaldo"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})

In [19]:
result['messages'][-1].content

'Player_2'

In [20]:
from langchain_core.messages import HumanMessage

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup()
    
    agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_4",
        checkpointer=checkpointer,
    )

    config = {"configurable": {"thread_id": "share_thread"}}

    human_msg = agent.update_state(
        config,
        {"messages": [HumanMessage(content="ddkjalkfj", name="agent_4")]}
    )
human_msg

{'configurable': {'thread_id': 'share_thread',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11bc61-0c3b-672c-8002-a80d0cfef822'}}

In [7]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    

    agent1_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_1"
    ]

    agent3_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_3_as_imposter"
    ]
    
    agent4_words = [
        msg.content for msg in request.messages
        if isinstance(msg, HumanMessage) and getattr(msg, "name", None) == "agent_4"
    ]

    #print("Agent1:", agent1_words)

    #print(request.messages)

   # print("Latest", words)

    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are a player in an Imposter social deduction game.
                    Players' words so far: {words}
                    Your secret keyword is {content}: 

                    Rules:
                    - Answer in ONE word only.
                    - Directly describe what it is or how it is used.
                    - Do NOT repeat previous words from other players.
                    - Be subtle. Do not make it too obvious.

                    """

    elif round_c in ["round_2", "round_3"]:
        return f"""
                Players' words so far: {words}
                Your keyword: {content}
                Think carefully:
                - Then give ONE new word.
                - Do NOT repeat any previous word.

                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_1's responses:
            {agent1_words}

            Player_3's responses:
            {agent3_words}

            Player_4's responses:
            {agent4_words}

            Based on these responses:
            - Guess who is the imposter (Player_1 or Player_3 or Player_4).
            - Return only the player name.
            """
    else:
        pass 
    return round_c

DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    
    agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_2",
        checkpointer=checkpointer,
    )
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Ronaldo"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})

In [8]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    
   # print("Latest", words)
    agent1_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_1"
    ]

    agent2_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_2"
    ]

    agent4_words = [
        msg.content for msg in request.messages
        if isinstance(msg, HumanMessage) and getattr(msg, "name", None) == "agent_4"
    ]



    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are an Imposter in a social deduction game.
            Players' words so far: {words}

            You DO NOT know the secret word.

            Rules:
            - Carefully analyze other players' responses.
            - Give ONE word only.
            - Do NOT repeat previous words.
            - Blend in naturally.
            - Try to guess what the keyword might be.

"""

    elif round_c in ["round_2", "round_3"]:
        return f"""
        Players' words so far:
        {words}

        You are the Imposter.

        - Analyze all clues.
        - Guess what the secret word might be.
        - Blend in naturally.
        - Give ONE new word.
        - Do NOT repeat previous words.

                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_1's responses:
            {agent1_words}

            Player_2's responses:
            {agent2_words}

            Player_4's responses:
            {agent4_words}

            Based on these responses:
            - Guess who is the imposter (Player_1 or Player_2 or Player_4).
            - Return only the player name.
            """
    else:
        pass 
    return round_c
DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    imposter_agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_3_as_imposter",
        checkpointer=checkpointer,
    )
    
    imposter_agent_rt = imposter_agent.invoke(
        {"messages": [{"role": "user", "content": "you are an imposter"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})
    

In [21]:
result['messages'][1].content

'ပန်း'

In [9]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent1 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "share_thread"
    }
}

agent1_history = checkpointer_agent1.get_tuple(config)
agent1_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b219-3796-6fb8-8008-dfdc2db65f3e'}}, checkpoint={'v': 4, 'id': '1f11b219-3796-6fb8-8008-dfdc2db65f3e', 'ts': '2026-03-08T19:04:04.727577+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000009.0.017627171835261524'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000008.0.8803342348538229'}}, 'channel_values': {'messages': [HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='8e7d208d-ef35-4a1e-88ca-5d56a61049e1'), AIMessage(content='Player_2', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 140, 'total_tokens': 298, 'completion_time': 0.334896693, 'completion_tokens_details': {'reasoning_tokens': 146}, 'prompt_time': 0.005604671, 'prompt_tokens_details': None, 'queue_time': 0.044906118, 'total_time': 0.340501364}, 'model_name': 'openai/gpt-

In [ ]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent2 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "2"
    }
}

agent2_history = checkpointer_agent2.get_tuple(config)
agent2_history

In [23]:
agent2_history[1]

{'v': 4,
 'id': '1f119731-5792-63a6-8001-1b08232782ca',
 'ts': '2026-03-06T15:42:29.738985+00:00',
 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000002.0.5432758134985926'},
  '__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.684758239359927'}},
 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='14e575df-038b-4bbe-b0a0-e5b59a7f8f66'),
   AIMessage(content='treat', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 236, 'prompt_tokens': 149, 'total_tokens': 385, 'completion_time': 0.504629861, 'completion_tokens_details': {'reasoning_tokens': 225}, 'prompt_time': 0.005667934, 'prompt_tokens_details': None, 'queue_time': 0.045236996, 'total_time': 0.510297795}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='age

In [12]:
agent1_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-4df2-6cca-8010-97e64aa504cf'}}, checkpoint={'v': 4, 'id': '1f119811-4df2-6cca-8010-97e64aa504cf', 'ts': '2026-03-06T17:22:41.684176+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000017.0.09720644422819291'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000016.0.1913759363839982'}}, 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='413be9ad-d00c-4980-ab2b-628f55b03809'), AIMessage(content='dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 149, 'total_tokens': 338, 'completion_time': 0.411281686, 'completion_tokens_details': {'reasoning_tokens': 178}, 'prompt_time': 0.006028855, 'prompt_tokens_details': None, 'queue_time': 0.044320374, 'total_time': 0.417310541}, 'model_name': 'openai/gpt-oss-

In [10]:
agent1_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b219-3796-6fb8-8008-dfdc2db65f3e'}}, checkpoint={'v': 4, 'id': '1f11b219-3796-6fb8-8008-dfdc2db65f3e', 'ts': '2026-03-08T19:04:04.727577+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000009.0.017627171835261524'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000008.0.8803342348538229'}}, 'channel_values': {'messages': [HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='8e7d208d-ef35-4a1e-88ca-5d56a61049e1'), AIMessage(content='Player_2', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 140, 'total_tokens': 298, 'completion_time': 0.334896693, 'completion_tokens_details': {'reasoning_tokens': 146}, 'prompt_time': 0.005604671, 'prompt_tokens_details': None, 'queue_time': 0.044906118, 'total_time': 0.340501364}, 'model_name': 'openai/gpt-

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b1c5-885f-691c-802b-d0053b432028'}}, checkpoint={'v': 4, 'id': '1f11b1c5-885f-691c-802b-d0053b432028', 'ts': '2026-03-08T18:26:38.340505+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000043.0.8988243808135317'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000042.0.17209459177276965'}}, 'channel_values': {'messages': [HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='15e270dc-151d-4163-91f7-eddb62e043c1'), AIMessage(content='Striker', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 156, 'total_tokens': 316, 'completion_time': 0.351882128, 'completion_tokens_details': {'reasoning_tokens': 149}, 'prompt_time': 0.005869817, 'prompt_tokens_details': None, 'queue_time': 0.045044643, 'total_time': 0.357751945}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cce9c-d865-7211-8da3-b1412b5b61aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 160, 'total_tokens': 316, 'output_token_details': {'reasoning': 149}}), HumanMessage(content='I dont know', additional_kwargs={}, response_metadata={}, id='dda3d4c5-733f-4f47-80cf-481311118bc6'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='efb82d2d-33a6-40fe-92f6-11678ea3ee2e'), AIMessage(content='Forward', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 232, 'prompt_tokens': 178, 'total_tokens': 410, 'completion_time': 0.483021872, 'completion_tokens_details': {'reasoning_tokens': 222}, 'prompt_time': 0.007490235, 'prompt_tokens_details': None, 'queue_time': 0.045690115, 'total_time': 0.490512107}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cce9d-1044-79d2-acdb-619f7e81fbc6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 232, 'total_tokens': 410, 'output_token_details': {'reasoning': 222}}), HumanMessage(content='you are an imposter', additional_kwargs={}, response_metadata={}, id='60b6aa7b-bcd0-4bad-8f4f-8e44e67b6432'), AIMessage(content='ဂိုး', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 196, 'total_tokens': 331, 'completion_time': 0.281157272, 'completion_tokens_details': {'reasoning_tokens': 124}, 'prompt_time': 0.007776581, 'prompt_tokens_details': None, 'queue_time': 0.044835539, 'total_time': 0.288933853}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_3_as_imposter', id='lc_run--019cce9d-3290-76b0-beb0-437dd9d859de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 135, 'total_tokens': 331, 'output_token_details': {'reasoning': 124}}), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='78d168be-6ed3-43ca-9717-cb0d252f029a'), AIMessage(content='Goalkeeper', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 174, 'total_tokens': 327, 'completion_time': 0.328831444, 'completion_tokens_details': {'reasoning_tokens': 142}, 'prompt_time': 0.019497014, 'prompt_tokens_details': None, 'queue_time': 0.064858236, 'total_time': 0.348328458}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cce9d-5c88-73b3-ae04-f09ad05439b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 174, 'output_tokens': 153, 'total_tokens': 327, 'output_token_details': {'reasoning': 142}}), HumanMessage(content='Messi', additional_kwargs={}, response_metadata={}, id='d0497c20-f1c6-41a9-91dc-9f6ae19039d2'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='7cd2df51-08b9-446f-83ac-b288c513dff5'), HumanMessage(content='you are an imposter', additional_kwargs={}, response_metadata={}, id='ae12092f-c007-4541-8b23-c46c9a646d50'), AIMessage(content='အလယ်တန်းကစားသမား', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 271, 'prompt_tokens': 230, 'total_tokens': 501, 'completion_time': 0.564707287, 'completion_tokens_details': {'reasoning_tokens': 252}, 'prompt_time': 0.009856731, 'prompt_tokens_details': None, 'queue_time': 0.045889309, 'total_time': 0.574564018}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_3_as_imposter', id='lc_run--019cce9d-8f00-7521-b284-b71ed83a39ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 271, 'total_tokens': 501, 'output_token_details': {'reasoning': 252}}), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='7e437f0d-b94f-473b-ad73-5d93aa15e1c0'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='b75e3cdf-e776-4225-8e5e-85735f336f84'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='e696f52b-a5d2-4f5f-8cc0-761ec013c52a'), AIMessage(content='CR7', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 255, 'total_tokens': 480, 'completion_time': 0.476112261, 'completion_tokens_details': {'reasoning_tokens': 214}, 'prompt_time': 0.011253248, 'prompt_tokens_details': None, 'queue_time': 0.046038942, 'total_time': 0.487365509}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cce9d-cf0a-7483-83fe-1f5c8fb7e91b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 255, 'output_tokens': 225, 'total_tokens': 480, 'output_token_details': {'reasoning': 214}}), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='a1b2b163-4004-402e-b445-be821c9df408'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='7e35f213-07e8-4646-a382-8c6f81c9b846'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='127f9036-f7e5-4c29-b5ca-c26a72b78768'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='57de3775-17d4-415d-b11d-0784915ee7a0'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='ef1b0e12-5322-4823-93aa-5b56f87f9ddc'), AIMessage(content='Player_3', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 519, 'prompt_tokens': 324, 'total_tokens': 843, 'completion_time': 1.0872275, 'completion_tokens_details': {'reasoning_tokens': 507}, 'prompt_time': 0.472980806, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.045808852, 'total_time': 1.560208306}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cce9e-62f1-7d62-af38-8bbffb8eb8de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 324, 'output_tokens': 519, 'total_tokens': 843, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 507}}), HumanMessage(content='Player_1', additional_kwargs={}, response_metadata={}, id='b725597a-3a51-4407-965b-c4074b9694a0'), HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='07910b22-d796-40f2-bd32-609c80891e65'), AIMessage(content='Player_1', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 438, 'prompt_tokens': 353, 'total_tokens': 791, 'completion_time': 0.91562136, 'completion_tokens_details': {'reasoning_tokens': 426}, 'prompt_time': 0.056001694, 'prompt_tokens_details': None, 'queue_time': 0.143181326, 'total_time': 0.971623054}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cce9e-aeef-7bb3-84d4-16b16f7d1516-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 353, 'output_tokens': 438, 'total_tokens': 791, 'output_token_details': {'reasoning': 426}}), HumanMessage(content='you are an imposter', additional_kwargs={}, response_metadata={}, id='527f5943-b55f-4bf4-a01e-309578a0fc03'), AIMessage(content='Player_4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 414, 'prompt_tokens': 367, 'total_tokens': 781, 'completion_time': 0.860062022, 'completion_tokens_details': {'reasoning_tokens': 402}, 'prompt_time': 0.01434268, 'prompt_tokens_details': None, 'queue_time': 0.04496694, 'total_time': 0.874404702}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_3_as_imposter', id='lc_run--019cce9e-c312-7af2-970f-38ba46a75e19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 367, 'output_tokens': 414, 'total_tokens': 781, 'output_token_details': {'reasoning': 402}}), HumanMessage(content='player', additional_kwargs={}, response_metadata={}, id='3f9cfe61-bbbb-4940-a3c6-73c57c39fdce')]}, 'channel_versions': {'messages': '00000000000000000000000000000045.0.3622217071377051', '__start__': '00000000000000000000000000000043.0.8988243808135317', 'branch:to:model': '00000000000000000000000000000044.0.5698269024339818'}, 'updated_channels': None}, metadata={'step': 43, 'source': 'update', 'parents': {}, 'lc_agent_name': 'agent_4'}, parent_config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b192-dd85-6c62-802a-f37c58bc2deb'}}, pending_writes=[])

In [ ]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent1 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "share_thread"
    }
}

agent1_history = checkpointer_agent1.get_tuple(config)
agent1_history

In [11]:
agent1 = []
agent2 = []
agent_3_as_imposter = []
agent_4 = []

for take_name_agent in agent1_history[1]['channel_values']['messages']:
    if take_name_agent.name == "agent_1":
        agent1.append(take_name_agent.content)
    elif take_name_agent.name == "agent_2":
        agent2.append(take_name_agent.content)
    elif take_name_agent.name == "agent_3_as_imposter":
        agent_3_as_imposter.append(take_name_agent.content)
    elif take_name_agent.name == "agent_4":
        agent_4.append(take_name_agent.content)
    else:
        pass
print("Agent1:" , agent1)
print("Agent2:" , agent2)
print("Agent3:", agent_3_as_imposter)
print("agent_4:", agent_4)

Agent1: ['Player_2']
Agent2: ['Player_1']
Agent3: ['Player_4']
agent_4: ['ddkjalkfj']


In [39]:
agent1_history[1]['channel_values']['messages']

[HumanMessage(content='Ronaldo', additional_kwargs={}, response_metadata={}, id='15e270dc-151d-4163-91f7-eddb62e043c1'),
 AIMessage(content='Striker', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 156, 'total_tokens': 316, 'completion_time': 0.351882128, 'completion_tokens_details': {'reasoning_tokens': 149}, 'prompt_time': 0.005869817, 'prompt_tokens_details': None, 'queue_time': 0.045044643, 'total_time': 0.357751945}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cce9c-d865-7211-8da3-b1412b5b61aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 160, 'total_tokens': 316, 'output_token_details': {'reasoning': 149}}),
 HumanMessage(content='I dont know', additional_kwargs={}, response_metadata={}, id='dda3d4c5-733f-4f47-80cf-4813

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-4df2-6cca-8010-97e64aa504cf'}}, checkpoint={'v': 4, 'id': '1f119811-4df2-6cca-8010-97e64aa504cf', 'ts': '2026-03-06T17:22:41.684176+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000017.0.09720644422819291'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000016.0.1913759363839982'}}, 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='413be9ad-d00c-4980-ab2b-628f55b03809'), AIMessage(content='dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 149, 'total_tokens': 338, 'completion_time': 0.411281686, 'completion_tokens_details': {'reasoning_tokens': 178}, 'prompt_time': 0.006028855, 'prompt_tokens_details': None, 'queue_time': 0.044320374, 'total_time': 0.417310541}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-13bf-7b43-b929-5e3db6ed3026-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 189, 'total_tokens': 338, 'output_token_details': {'reasoning': 178}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='f1fb0093-30e7-4dc6-aee7-a2e269984af3'), AIMessage(content='treat', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 165, 'total_tokens': 392, 'completion_time': 0.479108877, 'completion_tokens_details': {'reasoning_tokens': 216}, 'prompt_time': 0.006323071, 'prompt_tokens_details': None, 'queue_time': 0.046116659, 'total_time': 0.485431948}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-16a1-7891-a646-0fcb36c38a17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 227, 'total_tokens': 392, 'output_token_details': {'reasoning': 216}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='3b399e81-7d8c-49ea-a609-b81a112ff08e'), AIMessage(content='cake', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 231, 'prompt_tokens': 152, 'total_tokens': 383, 'completion_time': 0.500922681, 'completion_tokens_details': {'reasoning_tokens': 221}, 'prompt_time': 0.005771719, 'prompt_tokens_details': None, 'queue_time': 0.04583249, 'total_time': 0.5066944}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-298d-70c0-9d2f-1761c490f878-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 231, 'total_tokens': 383, 'output_token_details': {'reasoning': 221}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='c8614676-da6d-4eed-a173-26a0db3d898b'), AIMessage(content='pastry', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 169, 'total_tokens': 323, 'completion_time': 0.318490351, 'completion_tokens_details': {'reasoning_tokens': 143}, 'prompt_time': 0.006637686, 'prompt_tokens_details': None, 'queue_time': 0.045191942, 'total_time': 0.325128037}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-2cbd-7d03-b1ac-78cc8ac9697c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 169, 'output_tokens': 154, 'total_tokens': 323, 'output_token_details': {'reasoning': 143}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='103284ff-b1a0-429c-a74c-dc89644579c4'), AIMessage(content='pie', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 187, 'total_tokens': 349, 'completion_time': 0.344637025, 'completion_tokens_details': {'reasoning_tokens': 152}, 'prompt_time': 0.007685024, 'prompt_tokens_details': None, 'queue_time': 0.045595085, 'total_time': 0.352322049}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-40e9-7373-b0f7-086b5306c3f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 187, 'output_tokens': 162, 'total_tokens': 349, 'output_token_details': {'reasoning': 152}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='c64a0b08-d3ce-43d2-b717-97c4e41cecca'), AIMessage(content='cookie', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 204, 'total_tokens': 372, 'completion_time': 0.354691989, 'completion_tokens_details': {'reasoning_tokens': 158}, 'prompt_time': 0.008408945, 'prompt_tokens_details': None, 'queue_time': 0.045283894, 'total_time': 0.363100934}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-4386-7b13-91de-79d611429c8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 168, 'total_tokens': 372, 'output_token_details': {'reasoning': 158}})]}, 'channel_versions': {'messages': '00000000000000000000000000000018.0.9591710399123636', '__start__': '00000000000000000000000000000017.0.09720644422819291', 'branch:to:model': '00000000000000000000000000000018.0.9591710399123636'}, 'updated_channels': ['messages']}, metadata={'step': 16, 'source': 'loop', 'parents': {}, 'lc_agent_name': 'agent_2'}, parent_config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-48ea-680e-800f-67decfe182a4'}}, pending_writes=[])

In [6]:
from langchain_core.messages import AIMessage

ai_contents = [
    msg.content
    for msg in agent1_history[1]["channel_values"]["messages"]
    if isinstance(msg, AIMessage)
]

ai_contents

['dessert', 'treat', 'cake', 'pastry', 'pie', 'cookie']

In [19]:
from langchain_core.messages import AIMessage

ai_contents = [
    msg.content
    for msg in agent1_history[1]["channel_values"]["messages"]
    if isinstance(msg, AIMessage)
]

ai_contents

['dessert', 'treat', 'cake', 'pastry', 'pie', 'cookie']

In [ ]:
['pastry', 'dessert', 'baked', 'snack']


In [ ]:
['pastry', 'dessert', 'baked']
['treat', 'snack', 'pastry']

In [16]:
thread_id = "1"
checkpoint_deleter = checkpointer.delete_thread(thread_id)

In [17]:
checkpoint_deleter

In [18]:
checkpointer.get_tuple(config)

In [7]:
from langchain.messages import HumanMessage 

human_msg = HumanMessage(content="Playing Games", name="player")
human_msg

HumanMessage(content='Playing Games', additional_kwargs={}, response_metadata={}, name='player')

In [ ]:
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.checkpoint.base import empty_checkpoint
from langchain_core.messages import HumanMessage
import uuid

def save_human_message(content):
    with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
        checkpointer.setup()

        thread_id = "human_thread"
        checkpoint_ns = ""

        existing = checkpointer.get(
            {"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns}}
        )

        new_msg = HumanMessage(content=content)

        if existing:
            checkpoint = existing 

            checkpoint["channel_values"]["messages"].append(new_msg)

            current_version = int(checkpoint["channel_versions"]["messages"])
            new_version = str(current_version + 1)

            checkpoint["channel_versions"]["messages"] = new_version

        else:
            checkpoint = {
                **empty_checkpoint(),
                "id": str(uuid.uuid4()),
                "channel_values": {"messages": [new_msg]},
                "channel_versions": {"messages": "1"},
                "versions_seen": {},
            }

            new_version = "1"

        metadata = {
            "source": "input",
            "writes": {"messages": [new_msg]}
        }

        checkpointer.put(
            config={
                "configurable": {
                    "thread_id": thread_id,
                    "checkpoint_ns": checkpoint_ns,
                    "checkpoint_id": checkpoint["id"]
                }
            },
            checkpoint=checkpoint,
            metadata=metadata,
            new_versions={"messages": new_version}
        )


save_human_message("ugh... hoke lr")

In [38]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent2 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "share_thread"
    }
}

agent2_history = checkpointer_agent2.get_tuple(config)
agent2_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b0f7-9d83-6196-8000-cf24d40925ca'}}, checkpoint={'v': 4, 'id': '1f11b0f7-9d83-6196-8000-cf24d40925ca', 'ts': '2026-03-08T16:54:30.786692+00:00', 'versions_seen': {'__start__': {}}, 'channel_values': {'branch:to:model': None, 'messages': [HumanMessage(content='Mingalar pr', additional_kwargs={}, response_metadata={}, id='946999e5-28e3-4d20-9fd9-0e92f8f14e6f')]}, 'channel_versions': {'messages': '00000000000000000000000000000001.0.23362346629944108', 'branch:to:model': '00000000000000000000000000000001.0.23362346629944108'}, 'updated_channels': None}, metadata={'step': 0, 'source': 'update', 'parents': {}, 'lc_agent_name': 'agent_2'}, parent_config=None, pending_writes=[])

CheckpointTuple(config={'configurable': {'thread_id': 'human_thread', 'checkpoint_ns': '', 'checkpoint_id': '26c222ce-90be-422b-ad81-2419598a6b3b'}}, checkpoint={'v': 2, 'id': '26c222ce-90be-422b-ad81-2419598a6b3b', 'ts': '2026-03-08T16:33:53.732470+00:00', 'pending_sends': [], 'versions_seen': {}, 'channel_values': {'messages': [HumanMessage(content='Mingalapr pr', additional_kwargs={}, response_metadata={}), HumanMessage(content='ugh... hoke lr', additional_kwargs={}, response_metadata={})]}, 'channel_versions': {'messages': '2'}, 'updated_channels': None}, metadata={'source': 'input'}, parent_config={'configurable': {'thread_id': 'human_thread', 'checkpoint_ns': '', 'checkpoint_id': '26c222ce-90be-422b-ad81-2419598a6b3b'}}, pending_writes=[])

CheckpointTuple(config={'configurable': {'thread_id': 'human_thread', 'checkpoint_ns': '', 'checkpoint_id': '17ed5e65-ea53-44ea-8590-eb04e272c5e8'}}, checkpoint={'v': 2, 'id': '17ed5e65-ea53-44ea-8590-eb04e272c5e8', 'ts': '2026-03-08T15:53:50.718788+00:00', 'pending_sends': [], 'versions_seen': {}, 'channel_values': {'messages': [HumanMessage(content='Hello, I need help with X', additional_kwargs={}, response_metadata={})]}, 'channel_versions': {'messages': '1'}, 'updated_channels': None}, metadata={'step': 0, 'source': 'input'}, parent_config={'configurable': {'thread_id': 'human_thread', 'checkpoint_ns': '', 'checkpoint_id': '17ed5e65-ea53-44ea-8590-eb04e272c5e8'}}, pending_writes=[])

In [34]:
from langgraph.checkpoint.postgres import PostgresSaver

thread_id = "human_thread"

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup()

    checkpointer.delete_thread(thread_id)

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11b0f7-9d83-6196-8000-cf24d40925ca'}}, checkpoint={'v': 4, 'id': '1f11b0f7-9d83-6196-8000-cf24d40925ca', 'ts': '2026-03-08T16:54:30.786692+00:00', 'versions_seen': {'__start__': {}}, 'channel_values': {'branch:to:model': None, 'messages': [HumanMessage(content='Mingalar pr', additional_kwargs={}, response_metadata={}, id='946999e5-28e3-4d20-9fd9-0e92f8f14e6f')]}, 'channel_versions': {'messages': '00000000000000000000000000000001.0.23362346629944108', 'branch:to:model': '00000000000000000000000000000001.0.23362346629944108'}, 'updated_channels': None}, metadata={'step': 0, 'source': 'update', 'parents': {}, 'lc_agent_name': 'agent_2'}, parent_config=None, pending_writes=[])